# Profiling JAXQuantum functions

JAX dispatch is asynchronous, so ordinary wall-clock timing can measure dispatch rather than execution. JAXQuantum's profiling helpers synchronize every output leaf and report lowering, compilation, first execution, warmed execution, compiler memory, StableHLO, cost estimates, and precision tradeoffs.

In [ ]:
from pprint import pprint

import jax
import jax.numpy as jnp
import jaxquantum as jqt

print(f"JAX {jax.__version__} on {jax.default_backend()}: {jax.devices()}")

## Example function

We will profile a small oscillator calculation that constructs a displacement, applies it to a coherent state, and returns Fock-state probabilities. Profiling functions should return JAX arrays or PyTrees of arrays so execution can be synchronized.

In [ ]:
N = 24
state = jqt.coherent(N, 1.2)
beta = jnp.asarray(0.05)


def displaced_populations(psi, displacement):
    displaced = jqt.displace(N, displacement) @ psi
    return jnp.abs(displaced.data) ** 2

## Aggregate report

`benchmark_jax_function` is the usual entry point. `cold_total` is lowering + compilation + first execution; warmed statistics reuse the compiled executable. The reported compiler memory describes executable buffers rather than total process memory.

In [ ]:
report = jqt.benchmark_jax_function(
    displaced_populations,
    state,
    beta,
    iterations=8,
    warmup=1,
)

print("Timing (seconds)")
pprint(report["timings_s"])
print("\nCompiled memory (bytes)")
pprint(report["memory_bytes"])
print("\nStableHLO size", report["hlo"])
interesting_costs = {
    key: value
    for key, value in report["cost_analysis"].items()
    if key in {"flops", "transcendentals", "bytes accessed"}
}
print("Cost estimates")
pprint(interesting_costs)

## Individual HLO and memory helpers

Use the individual helpers when you do not need a complete benchmark. `jax_hlo` returns StableHLO text, `lower_jax_function` exposes JAX's lowered object, `jax_memory_stats` reads compiler buffer estimates, and `jax_device_memory_stats` queries each device allocator when the backend supports it.

In [ ]:
hlo = jqt.jax_hlo(displaced_populations, state, beta)
print("First StableHLO lines:")
print("\n".join(hlo.splitlines()[:12]))

compiled = jqt.lower_jax_function(
    displaced_populations, state, beta
).compile()
print("\nCompiled memory:")
pprint(jqt.jax_memory_stats(compiled))
print("\nDevice allocator snapshots:")
pprint(jqt.jax_device_memory_stats())

## Precision comparison

`compare_precision=True` profiles both float64/complex128 and float32/complex64, then compares output accuracy, speed, and compiled memory. Both modes run regardless of the current precision, and the original process-wide `jax_enable_x64` setting is restored afterward.

In [ ]:
original_x64 = jax.config.x64_enabled
precision_report = jqt.benchmark_jax_function(
    displaced_populations,
    state,
    beta,
    compare_precision=True,
    iterations=5,
    warmup=1,
)
assert jax.config.x64_enabled == original_x64

print("Accuracy loss in single precision")
pprint(precision_report["accuracy"])
print("\nSingle-versus-double ratios")
pprint(precision_report["single_vs_double"])

## Practical notes

- Compare repeated runs: cold compilation and allocator state vary with caches and other live arrays.
- Use warmed medians rather than dispatch time for steady-state performance.
- Compiler memory is best for function-to-function comparisons; allocator snapshots include inputs, caches, and unrelated live allocations.
- Pass `include_hlo=True` to the aggregate helper only when the full StableHLO text belongs in the report.
- Precision is process-global in JAX, so do not change it concurrently from another thread.